# Reconsider ongoing goals

A proactive agent pursues a standing goal as time and circumstances change. This notebook uses synthetic meetings and local tools. No model calls, credentials or external accounts are used.

Install the repository and `examples/proactive/requirements.txt` in your kernel environment. See [the tutorial](../README.md) for the conceptual explanation, diagrams, citations and limitations.

In [ ]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/noema').is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from examples.proactive.lessons import run_lesson

level = 3

## Read the actual policy

These functions are shared business code, not framework-provided behavior. Source is displayed so a small invocation does not hide its implementation.

In [ ]:
import inspect

from examples.proactive.domain import Tools, proposals

print(inspect.getsource(proposals))
print(inspect.getsource(Tools.invoke))

## Run the Noema implementation

The scenario asserts useful action and appropriate silence.

In [ ]:
from IPython.display import display

result = await run_lesson('noema', level)
display(result['calls'])
display(result['notices'])
assert len(result['notices']) == 1

## Inspect the SDK bridge

NoemaLesson is a tutorial adapter, not an SDK API. Its source uses real ActionIntent, AutonomousAgent, NoemaSystem, policy and capability APIs.

In [ ]:
from examples.proactive.noema_agent import NoemaLesson

print(inspect.getsource(NoemaLesson))

## Reproduce with ADK and LangGraph

Each equivalent executes its own framework. Compare tool calls, notices and final state, not incompatible raw event formats.

In [ ]:
equivalents = {}
for framework in ('adk', 'langgraph'):
    other = await run_lesson(framework, level)
    for key in ('calls', 'notices', 'world'):
        assert other[key] == result[key], (framework, key)
    equivalents[framework] = {'calls': len(other['calls']), 'notices': len(other['notices'])}
display(equivalents)

## Change the circumstances

Run an independent case where a meeting is cancelled before its window. Change `cancelled` to False to observe how that changes eligibility.

In [ ]:
tools = Tools({'design:v1': 'A synthetic agenda'})
agent = await NoemaLesson(level, tools).start()
cancelled = {'start': 10, 'revision': 1, 'importance': 5, 'cancelled': True}
try:
    await agent.tick(now=4, meetings={'design': cancelled})
    await agent.tick(now=5)
    assert tools.notices == []
    display(tools.calls)
finally:
    await agent.close()

## Measure instead of assuming an advantage

Physical line counts and AST statement counts measure different things. The shared policy, tools and host code count toward every implementation. Native traces show a domain-semantic benefit in Noema; this does not establish greater reliability or fewer source lines.

In [ ]:
from examples.proactive.comparison import compare

measurement, traces = await compare()
display(measurement['code_totals'])
display(measurement['native_noema_records_in_medium_lesson'])
causal_key = ('native_noema_authorization_and_success_have_observed_causal_parents')
assert measurement[causal_key]

Process persistence is tested separately with `python -m unittest tests.test_proactive_examples -v`. A notebook kernel remaining alive is not process-restart evidence. Mid-effect crashes and durable timer rearming are outside this tutorial's demonstrated guarantees.